# Task 1 — Get the Data Into a Database

**Qafza Tech — MLOps Engineering Training 2026/2027**

**Goal of this notebook:** download the Olist Brazilian E-Commerce dataset, load every table into a real PostgreSQL database, and prove it all works with a few queries and a join — no modeling or EDA yet, that's for later tasks.

**What we'll do, step by step:**
1. Set up our tools and config
2. Download the raw CSV files from Kaggle
3. Take a first peek at the raw data with pandas
4. Connect to our PostgreSQL database (running in Docker)
5. Load each CSV into its own table
6. Test everything with queries and a join
7. Recap the problem we're solving, so it's fresh going into Task 2

> 💡 **Tip:** run the cells top to bottom on your first pass. Nothing here is destructive — tables are recreated cleanly every run, so you can re-run this notebook as many times as you want.


## 1. Setup & Configuration

Just our imports and the database connection settings. We load secrets from a `.env` file so nothing sensitive ends up hardcoded in the notebook.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Find the repository root from the current working directory.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / ".env.example").exists() and (candidate / "data").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find the repository root containing .env.example and data/")

load_dotenv(REPO_ROOT / ".env")

DB_USER = os.getenv("POSTGRES_USER", "olist_user")
DB_PASS = os.getenv("POSTGRES_PASSWORD", "olist_pass")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "olist_db")

RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Config loaded. Target database:", DB_NAME, "@", f"{DB_HOST}:{DB_PORT}")
print("Active raw data folder:", RAW_DATA_DIR)

Config loaded. Target database: olist_db @ localhost:5432


## 2. Download the Dataset

We're using [`kagglehub`](https://github.com/Kaggle/kagglehub) so the download is one line of Python — no manual zip-file wrangling.

**Before running this cell, you need a Kaggle API token:**
1. Go to kaggle.com → your profile picture → **Settings**
2. Scroll to **API** → click **Create New Token** — this downloads `kaggle.json`
3. Either:
   - place `kaggle.json` in `~/.kaggle/kaggle.json`, **or**
   - set `KAGGLE_USERNAME` and `KAGGLE_KEY` in your `.env` file (values are inside `kaggle.json`)

If you'd rather do it manually: download the ZIP from the [Kaggle dataset page](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), unzip it, and drop the CSVs directly into `data/raw/`. Either way, the rest of the notebook works the same.

In [ ]:
import shutil
import kagglehub

# Download through KaggleHub, then copy the CSVs into the project's durable data/raw folder.
dataset_path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
for source_path in dataset_path.glob("*.csv"):
    shutil.copy2(source_path, RAW_DATA_DIR / source_path.name)

print("Dataset copied to:", RAW_DATA_DIR)
print()
print("Files found:")
for f in sorted(RAW_DATA_DIR.glob("*.csv")):
    print(" -", f.name)

100%|██████████| 42.6M/42.6M [00:04<00:00, 9.30MB/s]

Extracting files...


Dataset downloaded to: C:\Users\SKY\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2

Files found:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


## 3. First Look at the Raw Data

Before touching the database, let's confirm the files actually load and look the way the dataset description says they should. This is *not* EDA (that's a later task) — just a sanity check.

We map each CSV to a clean table name we'll reuse when loading into Postgres.

In [ ]:
# CSV filename -> the table name we'll use in Postgres
TABLE_MAP = {
    "olist_customers_dataset.csv": "olist_customers",
    "olist_orders_dataset.csv": "olist_orders",
    "olist_order_items_dataset.csv": "olist_order_items",
    "olist_order_payments_dataset.csv": "olist_order_payments",
    "olist_order_reviews_dataset.csv": "olist_order_reviews",
    "olist_products_dataset.csv": "olist_products",
    "olist_sellers_dataset.csv": "olist_sellers",
    "olist_geolocation_dataset.csv": "olist_geolocation",
    "product_category_name_translation.csv": "product_category_translation",
}

dataframes = {}

for csv_name, table_name in TABLE_MAP.items():
    csv_path = RAW_DATA_DIR / csv_name
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Missing {csv_name} in {RAW_DATA_DIR}. Run the download/copy cell first."
        )
    df = pd.read_csv(csv_path)
    dataframes[table_name] = df
    print(f"{table_name:32s} -> {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols")

olist_customers                  ->  99,441 rows x  5 cols
olist_orders                     ->  99,441 rows x  8 cols
olist_order_items                -> 112,650 rows x  7 cols
olist_order_payments             -> 103,886 rows x  5 cols
olist_order_reviews              ->  99,224 rows x  7 cols
olist_products                   ->  32,951 rows x  9 cols
olist_sellers                    ->   3,095 rows x  4 cols
olist_geolocation                -> 1,000,163 rows x  5 cols
product_category_translation     ->      71 rows x  2 cols


In [4]:
# Quick peek at the two most important tables for our problem: orders + items
display(dataframes["olist_orders"].head(3))
display(dataframes["olist_order_items"].head(3))


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


## 4. Connect to PostgreSQL (via Docker)

Make sure the database is running before you continue:

```bash
docker compose up -d
```

That spins up a Postgres 16 container (and a pgAdmin UI at `http://localhost:5050` if you want a GUI to browse the tables).

We use **SQLAlchemy** to create a connection engine — this is what lets pandas talk to Postgres directly.

In [5]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Test the connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print("Connected! Postgres version:")
    print(result.fetchone()[0])


Connected! Postgres version:
PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 5. Load Tables Into the Database

We load every DataFrame straight into Postgres with `to_sql()`. `if_exists="replace"` means you can re-run this notebook safely — it always starts from a clean set of tables instead of appending duplicates.

In [6]:
for table_name, df in dataframes.items():
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=5000)
    print(f"Loaded {table_name:32s} ({len(df):,} rows)")

print()
print("All tables loaded.")


Loaded olist_customers                  (99,441 rows)
Loaded olist_orders                     (99,441 rows)
Loaded olist_order_items                (112,650 rows)
Loaded olist_order_payments             (103,886 rows)
Loaded olist_order_reviews              (99,224 rows)
Loaded olist_products                   (32,951 rows)
Loaded olist_sellers                    (3,095 rows)
Loaded olist_geolocation                (1,000,163 rows)
Loaded product_category_translation     (71 rows)

All tables loaded.


## 6. Test It — Queries & Joins

First, confirm every table actually made it into the database and has the right row count.

In [7]:
with engine.connect() as conn:
    tables = conn.execute(text('''
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    ''')).fetchall()

print("Tables in the database:")
for t in tables:
    count = conn.execute(text(f'SELECT COUNT(*) FROM "{t[0]}"')).scalar() if False else None

with engine.connect() as conn:
    for (table_name,) in tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM "{table_name}"')).scalar()
        print(f"  {table_name:32s} {count:>8,} rows")


Tables in the database:
  olist_customers                    99,441 rows
  olist_geolocation                1,000,163 rows
  olist_order_items                 112,650 rows
  olist_order_payments              103,886 rows
  olist_order_reviews                99,224 rows
  olist_orders                       99,441 rows
  olist_products                     32,951 rows
  olist_sellers                       3,095 rows
  product_category_translation           71 rows


Now the real test — a **join across tables**. This is exactly the kind of query we'll rely on constantly once we start building the ML dataset: pulling an order together with its customer and its items.

In [8]:
join_query = text('''
    SELECT
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_estimated_delivery_date,
        o.order_delivered_customer_date,
        c.customer_city,
        c.customer_state,
        oi.product_id,
        oi.price,
        oi.freight_value
    FROM olist_orders o
    JOIN olist_customers c   ON o.customer_id = c.customer_id
    JOIN olist_order_items oi ON o.order_id = oi.order_id
    LIMIT 10;
''')

pd.read_sql(join_query, engine)


,order_id,order_status,order_purchase_timestamp,order_estimated_delivery_date,order_delivered_customer_date,customer_city,customer_state,product_id,price,freight_value
0,001ac194d4a326a6fa99b581e9a3d963,delivered,2018-07-04 11:39:11,2018-07-17 00:00:00,2018-07-12 17:48:49,cubatao,SP,dbaee28f4ee64465838a229582d77520,54.00,8.54
1,001b76dd48a5b1eee3e87778daa40df8,delivered,2018-03-26 17:42:53,2018-04-27 00:00:00,2018-04-06 18:36:23,santa salete,SP,dbb67791e405873b259e4656bf971246,81.99,13.01
2,001c85b5f68d2be0cb0797afc9e8ce9a,delivered,2017-11-24 19:19:18,2017-12-14 00:00:00,2017-12-22 18:37:40,sao paulo,SP,84f456958365164420cfc80fbe4c7fab,99.00,13.71
3,00010242fe8c5a6d1ba2dd792cb16214,delivered,2017-09-13 08:59:02,2017-09-29 00:00:00,2017-09-20 23:43:48,campos dos goytacazes,RJ,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29
4,00018f77f2f0320c557190d7a144bdd3,delivered,2017-04-26 10:53:06,2017-05-15 00:00:00,2017-05-12 16:04:24,santa fe do sul,SP,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93
5,000229ec398224ef6ca0657da4fc703e,delivered,2018-01-14 14:33:31,2018-02-05 00:00:00,2018-01-22 13:19:16,para de minas,MG,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87
6,00024acbcdf0a6daa1e931b038114c75,delivered,2018-08-08 10:00:35,2018-08-20 00:00:00,2018-08-14 13:32:39,atibaia,SP,7634da152a4610f1595efa32f14722fc,12.99,12.79
7,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,2017-02-04 13:57:51,2017-03-17 00:00:00,2017-03-01 16:42:31,varzea paulista,SP,ac6c3623068f30de03045865e4e10089,199.90,18.14
8,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,2017-05-15 21:42:34,2017-06-06 00:00:00,2017-05-22 13:44:35,uberaba,MG,ef92defde845ab8450f9d70c526ef70f,21.90,12.69
9,00054e8431b9d7675808bcb819fb4a32,delivered,2017-12-10 11:53:48,2018-01-04 00:00:00,2017-12-18 22:03:38,guararapes,SP,8d4f2bb7e93e6710a28f34fa83ee7d28,19.90,11.85


**A note on this join:** notice that a single `order_id` can appear more than once here — that's because `olist_order_items` has **one row per item**, not one row per order. If we joined `payments` in too (also multiple rows per order), row counts would multiply again.

This is exactly the leakage/duplication trap the task brief warns about: before building a one-row-per-order ML table, we'll need to `GROUP BY order_id` and aggregate items and payments *first*, then join.

## 7. Understanding the Problem We're Solving

Before closing out, let's translate the business question into something concrete using the columns we now have.

In [9]:
# A quick, exploratory-only look at what "late" actually means in this data.
# This is NOT feature engineering yet — just confirming the target concept makes sense.

target_check = text('''
    SELECT
        order_id,
        order_status,
        order_estimated_delivery_date,
        order_delivered_customer_date,
        (order_delivered_customer_date > order_estimated_delivery_date) AS is_late
    FROM olist_orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL
    LIMIT 10;
''')

df_target = pd.read_sql(target_check, engine)
display(df_target)

late_rate = pd.read_sql(text('''
    SELECT
        AVG(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) AS late_rate,
        COUNT(*) AS total_delivered_orders
    FROM olist_orders
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date IS NOT NULL;
'''), engine)

display(late_rate)


,order_id,order_status,order_estimated_delivery_date,order_delivered_customer_date,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-18 00:00:00,2017-10-10 21:25:13,False
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-08-13 00:00:00,2018-08-07 15:27:45,False
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-09-04 00:00:00,2018-08-17 18:06:29,False
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-12-15 00:00:00,2017-12-02 00:28:42,False
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-26 00:00:00,2018-02-16 18:17:02,False
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-08-01 00:00:00,2017-07-26 10:57:55,False
6,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-06-07 00:00:00,2017-05-26 12:55:51,False
7,76c6e866289321a7c93b82b54852dc33,delivered,2017-03-06 00:00:00,2017-02-02 14:08:10,False
8,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,2017-08-23 00:00:00,2017-08-16 17:14:30,False
9,e6ce16cb79ec1d90b1da9085a6118aeb,delivered,2017-06-07 00:00:00,2017-05-29 11:18:31,False


,late_rate,total_delivered_orders
0,0.081124,96470


**Problem recap:**

- **Target (`is_late`):** `1` if `order_delivered_customer_date > order_estimated_delivery_date`, else `0`. Only defined for orders with `order_status = 'delivered'`.
- **Leakage risk:** `order_delivered_customer_date` and everything from `olist_order_reviews` only exist *after* the order has already arrived — they can't be used as model **inputs**, only to compute the label itself.
- **What we *can* use as features later:** anything known at (or before) purchase/shipping time — product category, weight/dimensions, freight value, payment type/installments, customer & seller location, day-of-week of purchase, etc.
- **Row granularity trap:** `order_items` and `order_payments` are multi-row-per-order — they must be aggregated to one row per `order_id` before joining into a final ML table (Task 2+).

## ✅ Done Checklist

- [x] Database running locally with the data inside
- [x] Queried the tables and joined two of them
- [x] Understand the tables and how they relate (`order_id`, `customer_id`, `product_id`, `seller_id`)
- [x] Understand the problem we're solving (late-delivery classification) and where leakage could sneak in


